In [1]:
import os
import requests
from typing import TypedDict, List
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from datetime import datetime

load_dotenv()

True

In [2]:
class State(TypedDict, total=False):
    city: str
    attractions: List[str]
    weather: str
    itinerary: str

In [3]:
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY is not set. Add it to .env or your environment.")

# Initialize the ChatGroq instance
llm = ChatGroq(model="qwen/qwen3-32b", api_key=api_key, temperature=0)

In [4]:
def suggest_attractions(state: State):
    prompt = ChatPromptTemplate.from_messages([
        ("human", "List 5 must-see attractions in {city}.")
    ])
    messages = prompt.format_messages(city=state["city"])
    response = llm.invoke(messages)
    text = (response.content or "").strip()
    attractions = [line.strip() for line in text.split("\n") if line.strip()]

    return {"attractions": attractions}

In [5]:
OPENWEATHER_API_KEY = os.getenv("OPEN_WEATHER_API_KEY") or os.getenv("OPENWEATHER_API_KEY")

@tool
def get_weather(city: str) -> str:
    """Fetch the current weather for a city."""
    if not OPENWEATHER_API_KEY:
        return "Weather unavailable: API key not set (OPEN_WEATHER_API_KEY or OPENWEATHER_API_KEY in .env)."
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units=metric"
    r = requests.get(url)
    data = r.json()
    if r.status_code != 200 or data.get("cod") != 200:
        return "Weather information not available for this city."
    city_name = data["name"]
    country = data["sys"]["country"]

    temp = data["main"]["temp"]
    feels_like = data["main"]["feels_like"]
    humidity = data["main"]["humidity"]
    pressure = data["main"]["pressure"]

    description = data["weather"][0]["description"]

    wind_speed = data["wind"]["speed"]
    clouds = data["clouds"]["all"]
    visibility = data["visibility"] / 1000  # km

    sunrise = datetime.fromtimestamp(data["sys"]["sunrise"]).strftime("%H:%M")
    sunset = datetime.fromtimestamp(data["sys"]["sunset"]).strftime("%H:%M")

    return f"""
        Weather in {city_name}, {country}
        
        Temperature: {temp}°C (feels like {feels_like}°C)
        Condition: {description}
        Cloud Coverage: {clouds}%
        
        Humidity: {humidity}%
        Wind Speed: {wind_speed} m/s
        Visibility: {visibility} km
        Pressure: {pressure} hPa
        
        Sunrise: {sunrise}
        Sunset: {sunset}
        """

In [6]:
def check_weather(state: State):
    city = state["city"]
    weather_data = get_weather.invoke({"city": city})

    prompt = ChatPromptTemplate.from_messages([
        ("human", "Here is the weather data for {city}:\n\n{weather}\n\nWrite a short weekend weather summary for travelers.")
    ])
    messages = prompt.format_messages(city=city, weather=weather_data)
    response = llm.invoke(messages)
    summary = (response.content or "").strip()
    return {"weather": summary}

In [7]:
def create_itinerary(state: State):
    prompt = ChatPromptTemplate.from_messages([
        ("human", "Create a 1-day itinerary for visiting {city} based on these attractions: {attractions} and the following weather: {weather}. Summarize it in 3-4 sentences.")
    ])
    attractions_str = ", ".join(state.get("attractions") or [])
    messages = prompt.format_messages(
        city=state["city"],
        attractions=attractions_str,
        weather=state.get("weather", "")
    )
    response = llm.invoke(messages)
    itinerary = (response.content or "").strip()
    return {"itinerary": itinerary}

In [8]:
workflow = StateGraph(State)

workflow.add_node("attractions_node", suggest_attractions)
workflow.add_node("weather_node", check_weather)
workflow.add_node("itinerary_node", create_itinerary)

workflow.set_entry_point("attractions_node")

workflow.add_edge("attractions_node", "weather_node")
workflow.add_edge("weather_node", "itinerary_node")
workflow.add_edge("itinerary_node", END)

app = workflow.compile()

In [9]:
sample_input = {"city": "Tunis"}

result = app.invoke(sample_input)

print("Top Attractions:", result["attractions"])
print("\nWeather Summary:", result["weather"])
print("\nItinerary:", result["itinerary"])

Top Attractions: ['<think>', "Okay, so I need to list five must-see attractions in Tunis. Let me start by recalling what I know about Tunis. Tunis is the capital of Tunisia, right? It's a city with a mix of ancient and modern elements. I remember that there's a medina, which is a historic part of the city. The medina of Tunis is a UNESCO World Heritage Site, so that's probably a good one. Then there's the Bardo Museum, which I think is famous for its Roman mosaics. That should be another one.", "What else? Maybe the Zitouna Mosque? I think that's a significant religious site. Then there's the Carthage ruins, which are ancient. Wait, are the Carthage ruins in Tunis or nearby? I think they're just outside the city, maybe a short trip away. But since the question is about Tunis, maybe they count. Alternatively, maybe the Habib Bourguiba Avenue is a notable area. Or the Avenue de Paris, which is a main street. But I'm not sure if that's considered a must-see.", "Another thought: the Sousse